# Binary Logistic Regression from Scratch

This project implements binary logistic regression from the ground up using NumPy, following the scikit-learn API pattern (`fit`, `predict`, `predict_proba`). The implementation uses vectorized gradient descent on the log-likelihood objective and is validated against scikit-learn's `LogisticRegression` on synthetic data.

An ablation study then examines how the learning rate and number of gradient descent iterations jointly affect training accuracy,illustrating how hyperparameter choices influence convergence behavior.

## 1. scikit-learn Baseline

Synthetic data (1000 samples, 20 features) is generated and split 70/30. A scikit-learn logistic regression model without regularization is trained as a reference point for the custom implementation.

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

np.random.seed(2024)
n = 1000
features = 20

X = np.random.normal(size=(n, features))
weights = np.random.normal(size=features)
probs = sigmoid(X @ weights + np.random.normal(scale=0.01))
y = np.random.binomial(n=1, p=probs)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2024)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(penalty=None, max_iter=1000)
model.fit(X_train, y_train)

print('scikit-learn LogisticRegression (no regularization)')
print(f'  Train accuracy: {accuracy_score(y_train, model.predict(X_train)):.4f}')
print(f'  Test accuracy:  {accuracy_score(y_test, model.predict(X_test)):.4f}')

Accurary on training set: 0.8714285714285714
Accurary on test set: 0.88 



## 2. Custom Implementation: `BinaryLogisticRegression`

The class below implements binary logistic regression using vectorized NumPy operations. Key design decisions:

- **Weights** are initialized from a standard normal distribution.
- **`predict_proba`** computes the sigmoid of the dot product between weights and input features.
- **`predict`** thresholds predicted probabilities at 0.5.
- **`fit`** runs gradient descent on the log-likelihood loss for `iters` iterations. The gradient for weight $w_j$ averaged over the training set is:

$$\nabla_j = \frac{1}{n} \sum_{i=1}^{n} (a_i - y_i)\, x_{ij}$$

The weight update rule is: $\vec{w}' = \vec{w} - \eta\, \vec{\nabla}$

All operations are vectorized — no loops over individual data points.

In [6]:
class BinaryLogisticRegression:
    def __init__(self, lr=0.1, iters=100, random_state=2024):
        self.lr = lr
        self.iters = iters
        self.random_state = random_state
        self.w = np.random.normal(0, 1, len(X[0]))

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-z))

    def predict_proba(self, X):
        """Predict probability of class 1 for each row in X."""
        z = X @ self.w
        return self.sigmoid(z)

    def predict(self, X):
        """Predict class label (0 or 1) for each row in X."""
        probs = self.predict_proba(X)
        return (probs >= 0.5).astype(int)

    def fit(self, X, y, verbose=False):
        """
        Fit the model using gradient descent on the log-likelihood.

        Parameters
        ----------
        X : array-like, shape = [n_examples, n_features]
        y : array-like, shape = [n_examples], values in {0, 1}
        """
        for i in range(self.iters):
            probs = self.predict_proba(X)
            gradient = ((probs - y) @ X) / X.shape[0]
            self.w = self.w - self.lr * gradient

            if verbose:
                print(f'Iteration {i}: gradient norm = {np.linalg.norm(gradient):.6f}')

        return


## 3. Validation Against scikit-learn

The custom implementation is trained with the same settings and compared to the scikit-learn baseline. Both models should achieve similar accuracy on train and test sets.

In [8]:
model_diy = BinaryLogisticRegression(lr=0.1, iters=1000, random_state=2024)
model_diy.fit(X_train, y_train)

print('BinaryLogisticRegression (custom, lr=0.1, iters=1000)')
print(f'  Train accuracy: {accuracy_score(y_train, model_diy.predict(X_train)):.4f}')
print(f'  Test accuracy:  {accuracy_score(y_test, model_diy.predict(X_test)):.4f}')

Accurary on training set: 0.8671428571428571
Accurary on testing set: 0.8766666666666667


## 4. Ablation Study: Learning Rate × Iterations

20 models are trained across all combinations of learning rate `lr ∈ {10, 1, 0.1, 0.01, 0.001}` and `iters ∈ {1, 5, 20, 100}`. Training accuracy (not test accuracy) is used for hyperparameter selection to avoid data leakage.

In [10]:
iters_list = [1, 5, 20, 100]
lr_list = [10, 1, 0.1, 0.01, 0.001]

for lr in lr_list:
    for iters in iters_list:
        m = BinaryLogisticRegression(lr=lr, iters=iters)
        m.fit(X_train, y_train)
        acc = accuracy_score(y_train, m.predict(X_train))
        # print(f'lr={lr}, iters={iters}: train accuracy={acc:.3f}')

### Results

| Learning Rate | 1 iteration | 5 iterations | 20 iterations | 100 iterations |
|:---:|:---:|:---:|:---:|:---:|
| **10** | 0.843 | 0.867 | 0.869 | 0.867 |
| 1 | 0.464 | 0.681 | 0.853 | 0.869 |
| 0.1 | 0.561 | 0.530 | 0.423 | 0.771 |
| 0.01 | 0.569 | 0.504 | 0.457 | 0.484 |
| 0.001 | 0.417 | 0.420 | 0.504 | 0.514 |

**Selected: `lr=10`**

`lr=10` achieves the highest training accuracy across all iteration counts, and reaches near-optimal performance in a single iteration — making it the most computationally efficient choice. Smaller learning rates (≤ 0.1) fail to converge even at 100 iterations, while `lr=10` shows stable convergence from the very first step.

This illustrates a key principle in gradient descent: an appropriately large learning rate can converge far faster than a conservative one, provided it does not overshoot and diverge.